# GPU-Native Deep CFR Validation

This notebook validates the device-native Deep CFR path against the existing hybrid batched backend. It keeps traversal records and replay on the GPU, uses fused fitting where available, and avoids inline exact evaluation during measured training.

The notebook intentionally separates correctness, throughput, and learning quality.

In [1]:
from pathlib import Path
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.deep_cfr import DeepCFRTrainer, DeviceReservoirBuffer
from liars_poker.algo.neural_cfr_gpu import GPUDeepCFRTraverser
from liars_poker.core import GameSpec, card_rank, generate_deck
from liars_poker.env import resolve_call_winner, rules_for_spec
from liars_poker.infoset import CALL
from liars_poker.policies.neural import compile_neural_to_dense
from liars_poker.training.deep_cfr import deep_cfr_timed_loop

assert torch.cuda.is_available(), 'This notebook is intended for a CUDA machine.'
device = 'cuda'
torch.set_float32_matmul_precision('high')

print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA runtime:', torch.version.cuda)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'GPU memory: {free_bytes / 2**30:.1f} / {total_bytes / 2**30:.1f} GiB free')

torch: 2.12.0+cu130
GPU: NVIDIA GeForce RTX 3090
CUDA runtime: 13.0
GPU memory: 22.5 / 23.6 GiB free


In [2]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'Trips'),
    suit_symmetry=True,
)
rules = rules_for_spec(spec)
print(spec.to_short_str(), 'claims:', len(rules.claims))

r4_s4_h3_hpt_ss claims: 12


## Static Correctness Checks

In [3]:
def hand_counts(hand, spec):
    counts = np.zeros(spec.ranks, dtype=np.float32)
    for card in hand:
        counts[card_rank(card, spec) - 1] += 1.0
    return counts


check_trainer = DeepCFRTrainer(
    spec,
    hidden_sizes=(32, 32),
    device=device,
    seed=11,
    advantage_train_steps=0,
    strategy_train_steps=0,
    traversal_backend='gpu_native',
    traversal_batch_size=16,
)
runner = GPUDeepCFRTraverser(check_trainer)

deck = list(generate_deck(spec))
rng = random.Random(123)
p1_hands, p2_hands = [], []
for _ in range(24):
    rng.shuffle(deck)
    p1_hands.append(tuple(sorted(deck[:spec.hand_size])))
    p2_hands.append(tuple(sorted(deck[spec.hand_size:2 * spec.hand_size])))

p1_counts = torch.tensor(np.stack([hand_counts(h, spec) for h in p1_hands]), device=device)
p2_counts = torch.tensor(np.stack([hand_counts(h, spec) for h in p2_hands]), device=device)
total_counts = p1_counts + p2_counts

claim_ids = torch.tensor(
    [0, 1, len(rules.claims) // 2, len(rules.claims) - 1] * 6,
    dtype=torch.long,
    device=device,
)
deal_ids = torch.arange(len(claim_ids), device=device)
gpu_values = runner._terminal_values(
    claim_ids,
    deal_ids,
    caller=1,
    traverser=0,
    total_counts=total_counts,
).cpu().numpy()

slow_values = []
for claim, deal_id in zip(claim_ids.cpu().tolist(), deal_ids.cpu().tolist()):
    winner = resolve_call_winner(
        spec,
        (claim, CALL),
        p1_hands[deal_id],
        p2_hands[deal_id],
    )
    slow_values.append(1.0 if winner == 'P1' else -1.0)

assert np.array_equal(gpu_values, np.asarray(slow_values, dtype=np.float32))
print('GPU terminal resolution matches Env resolution.')

GPU terminal resolution matches Env resolution.


In [4]:
smoke = DeepCFRTrainer(
    spec,
    hidden_sizes=(64, 64),
    device=device,
    seed=17,
    batch_size=256,
    advantage_train_steps=3,
    strategy_train_steps=2,
    advantage_positive_weight=0.5,
    advantage_buffer_capacity=20_000,
    strategy_buffer_capacity=20_000,
    traversal_backend='gpu_native',
    traversal_batch_size=32,
    validation_fraction=0.02,
)
smoke_record = smoke.run_iteration(traversals_per_player=64)
assert all(isinstance(buffer, DeviceReservoirBuffer) for buffer in smoke.advantage_buffers)
assert sum(smoke_record['new_advantage_records']) > 0
assert sum(smoke_record['new_strategy_records']) > 0
assert np.isfinite(smoke_record['advantage_loss']).all()
assert np.isfinite(smoke_record['strategy_loss']).all()
display(pd.json_normalize(smoke_record))

,iteration,advantage_loss,strategy_loss,advantage_buffer_sizes,strategy_buffer_sizes,advantage_records_seen,strategy_records_seen,new_advantage_records,new_strategy_records,timing.traversal_s,timing.advantage_training_s,timing.strategy_training_s
0,1,"[2.0062923431396484, 1.8553565740585327]","[1.58260178565979, 1.6282404661178589]","[61, 62]","[692, 745]","[61, 62]","[692, 745]","[61, 62]","[692, 745]",0.42997,0.100092,0.033115


In [5]:
# Distributional check: with identical zero-initialized networks, the two
# traversal implementations should estimate the same per-infoset regrets.
check_spec = GameSpec(
    ranks=2,
    suits=2,
    hand_size=1,
    claim_kinds=('RankHigh',),
    suit_symmetry=True,
)


def target_means(backend):
    trainer = DeepCFRTrainer(
        check_spec,
        hidden_sizes=(8,),
        device=device,
        seed=123,
        advantage_train_steps=0,
        strategy_train_steps=0,
        advantage_buffer_capacity=50_000,
        strategy_buffer_capacity=50_000,
        traversal_backend=backend,
        traversal_batch_size=256,
    )
    trainer.run_iteration(traversals_per_player=5_000)
    buffer = trainer.advantage_buffers[0]
    features = buffer.features[:buffer.size]
    targets = buffer.targets[:buffer.size]
    if torch.is_tensor(features):
        features = features.cpu().numpy()
        targets = targets.cpu().numpy()
    groups = {}
    for feature, target in zip(features, targets):
        groups.setdefault(tuple(feature.tolist()), []).append(target)
    return {key: np.mean(values, axis=0) for key, values in groups.items()}


hybrid_means = target_means('batched')
native_means = target_means('gpu_native')
common = sorted(set(hybrid_means) & set(native_means))
mean_differences = [
    np.max(np.abs(hybrid_means[key] - native_means[key])) for key in common
]
assert common and max(mean_differences) < 0.08
print('Maximum per-action regret-mean difference:', max(mean_differences))

Maximum per-action regret-mean difference: 0.00018942356


## Backend and GPU-Feature Timing

The first timing screen uses full online iterations, including traversal, replay insertion, advantage fitting, and strategy fitting. Exact evaluation is absent.

In [6]:
TIMING_REPEATS = 3
TIMING_TRAVERSALS = 512
TIMING_CONFIGS = [
    {
        'label': 'hybrid batched fp32',
        'backend': 'batched',
        'traversal_batch_size': 256,
        'device_replay': False,
        'amp_dtype': None,
        'compile_models': False,
    },
    {
        'label': 'gpu native fp32',
        'backend': 'gpu_native',
        'traversal_batch_size': 256,
        'device_replay': True,
        'amp_dtype': None,
        'compile_models': False,
    },
    {
        'label': 'gpu native bf16',
        'backend': 'gpu_native',
        'traversal_batch_size': 256,
        'device_replay': True,
        'amp_dtype': 'bfloat16',
        'compile_models': False,
    },
    {
        'label': 'gpu native bf16 compiled',
        'backend': 'gpu_native',
        'traversal_batch_size': 256,
        'device_replay': True,
        'amp_dtype': 'bfloat16',
        'compile_models': True,
    },
]


def make_trainer(config, seed):
    return DeepCFRTrainer(
        spec,
        hidden_sizes=(512, 512),
        device=device,
        seed=seed,
        batch_size=4096,
        advantage_train_steps=100,
        strategy_train_steps=50,
        advantage_positive_weight=0.5,
        advantage_buffer_capacity=300_000,
        strategy_buffer_capacity=300_000,
        validation_fraction=0.0,
        traversal_backend=config['backend'],
        traversal_batch_size=config['traversal_batch_size'],
        device_replay=config['device_replay'],
        fused_optimizer=True,
        amp_dtype=config['amp_dtype'],
        compile_models=config.get('compile_models', False),
    )


timing_rows = []
for config in TIMING_CONFIGS:
    for repeat in range(TIMING_REPEATS):
        try:
            trainer = make_trainer(config, seed=100 + repeat)
            # Warm up kernels, optimizers, replay allocation, and optional compilation.
            trainer.run_iteration(traversals_per_player=64)
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()
            record = trainer.run_iteration(traversals_per_player=TIMING_TRAVERSALS)
            torch.cuda.synchronize()
            wall_s = time.perf_counter() - start
            timing = record['timing']
            new_records = sum(record['new_advantage_records']) + sum(record['new_strategy_records'])
            timing_rows.append({
                'configuration': config['label'],
                'repeat': repeat,
                'wall s': wall_s,
                'traversal s': timing['traversal_s'],
                'advantage fit s': timing['advantage_training_s'],
                'strategy fit s': timing['strategy_training_s'],
                'records': new_records,
                'records / traversal s': new_records / timing['traversal_s'],
                'peak GPU GiB': torch.cuda.max_memory_allocated() / 2**30,
                'error': None,
            })
        except Exception as exc:
            timing_rows.append({
                'configuration': config['label'],
                'repeat': repeat,
                'error': repr(exc),
            })

timing_raw_df = pd.DataFrame(timing_rows)
timing_valid_df = timing_raw_df[timing_raw_df['error'].isna()].copy()
timing_summary_df = (
    timing_valid_df
    .groupby('configuration', as_index=False)
    .agg(
        median_wall_s=('wall s', 'median'),
        median_traversal_s=('traversal s', 'median'),
        median_advantage_fit_s=('advantage fit s', 'median'),
        median_strategy_fit_s=('strategy fit s', 'median'),
        median_records_per_traversal_s=('records / traversal s', 'median'),
        peak_GPU_GiB=('peak GPU GiB', 'max'),
    )
    .sort_values('median_wall_s')
)
display(timing_summary_df)
display(timing_raw_df)

,configuration,median_wall_s,median_traversal_s,median_advantage_fit_s,median_strategy_fit_s,median_records_per_traversal_s,peak_GPU_GiB
2,gpu native fp32,1.650089,0.164631,0.996239,0.497389,436291.112190,0.470642
1,gpu native bf16 compiled,1.804572,0.178280,1.008126,0.653073,407637.595233,0.620590
0,gpu native bf16,1.992315,0.170410,1.210832,0.604902,387090.622778,0.974319
3,hybrid batched fp32,2.558793,0.851509,1.036565,0.642903,72956.238329,0.143239


,configuration,repeat,wall s,traversal s,advantage fit s,strategy fit s,records,records / traversal s,peak GPU GiB,error
0,hybrid batched fp32,0,2.558793,0.851509,1.036565,0.669205,60354,70878.910742,0.143239,None
1,hybrid batched fp32,1,1.950219,0.807525,0.740886,0.400914,58914,72956.238329,0.143239,None
2,hybrid batched fp32,2,2.606003,0.909277,1.052406,0.642903,70323,77339.431271,0.143239,None
3,gpu native fp32,0,1.600925,0.164631,0.966975,0.468769,66139,401740.404143,0.298875,None
4,gpu native fp32,1,1.675619,0.175294,0.999499,0.500395,76479,436291.112190,0.298703,None
5,gpu native fp32,2,1.650089,0.156217,0.996239,0.497389,70242,449644.066976,0.470642,None
6,gpu native bf16,0,1.992315,0.170410,1.216710,0.604902,65964,387090.622778,0.631364,None
7,gpu native bf16,1,1.911266,0.163673,1.152117,0.595221,75189,459386.340874,0.803119,None
8,gpu native bf16,2,2.017305,0.196941,1.210832,0.609301,70389,357412.454470,0.974319,None
9,gpu native bf16 compiled,0,1.804572,0.178280,0.959043,0.666982,69049,387307.327110,0.277074,None


In [7]:
TRAVERSAL_BATCH_SIZES = [64, 128, 256, 512, 1024]
traversal_rows = []
for traversal_batch_size in TRAVERSAL_BATCH_SIZES:
    config = {
        'label': f'gpu native batch={traversal_batch_size}',
        'backend': 'gpu_native',
        'traversal_batch_size': traversal_batch_size,
        'device_replay': True,
        'amp_dtype': 'bfloat16',
        'compile_models': False,
    }
    trainer = make_trainer(config, seed=211)
    trainer.advantage_train_steps = 0
    trainer.strategy_train_steps = 0
    trainer.run_iteration(traversals_per_player=64)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    record = trainer.run_iteration(traversals_per_player=1024)
    torch.cuda.synchronize()
    records = sum(record['new_advantage_records']) + sum(record['new_strategy_records'])
    traversal_rows.append({
        'traversal batch size': traversal_batch_size,
        'traversal s': record['timing']['traversal_s'],
        'records': records,
        'records / s': records / record['timing']['traversal_s'],
        'peak GPU GiB': torch.cuda.max_memory_allocated() / 2**30,
    })

traversal_df = pd.DataFrame(traversal_rows)
display(traversal_df.sort_values('records / s', ascending=False))

,traversal batch size,traversal s,records,records / s,peak GPU GiB
4,1024,0.016460,26624,1.617520e+06,1.423212
3,512,0.031607,26624,8.423339e+05,1.251982
2,256,0.061968,26624,4.296377e+05,1.087236
1,128,0.123339,26624,2.158608e+05,0.924371
0,64,0.248672,26624,1.070648e+05,0.763682


## Fixed-Training-Time Online Comparison

Set the switch below after selecting the best traversal batch size. Training time excludes exact evaluation. Exact generated averaging is intentionally disabled because compiling the current policy every iteration would obscure the GPU comparison.

In [11]:
RUN_ONLINE_COMPARISON = True
TRAINING_MINUTES_PER_CONFIGURATION = 5
EVAL_CHUNK_SECONDS = 60
BEST_GPU_NATIVE_TRAVERSAL_BATCH = 1024  # replace from the traversal timing table

ONLINE_CONFIGS = [
    {
        'label': 'hybrid batched fp32',
        'backend': 'batched',
        'traversal_batch_size': BEST_GPU_NATIVE_TRAVERSAL_BATCH,
        'device_replay': False,
        'amp_dtype': None,
    },
    {
        'label': 'gpu native fp32',
        'backend': 'gpu_native',
        'traversal_batch_size': BEST_GPU_NATIVE_TRAVERSAL_BATCH,
        'device_replay': True,
        'amp_dtype': None,
    },
    {
        'label': 'gpu native bf16',
        'backend': 'gpu_native',
        'traversal_batch_size': BEST_GPU_NATIVE_TRAVERSAL_BATCH,
        'device_replay': True,
        'amp_dtype': 'bfloat16',
    },
]


def exact_metrics(trainer):
    learned_dense = compile_neural_to_dense(trainer.average_policy())
    _, learned_meta = best_response_dense(
        spec, learned_dense, debug=False, store_state_values=False
    )
    learned_p1, learned_p2 = learned_meta['computer'].exploitability()

    current_dense = trainer.current_policy_dense()
    _, current_meta = best_response_dense(
        spec, current_dense, debug=False, store_state_values=False
    )
    current_p1, current_p2 = current_meta['computer'].exploitability()
    return {
        'learned average exploitability': learned_p1 + learned_p2 - 1.0,
        'current exploitability': current_p1 + current_p2 - 1.0,
    }


online_training_rows = []
online_eval_rows = []
online_trainers = {}

if RUN_ONLINE_COMPARISON:
    chunks = int(60 * TRAINING_MINUTES_PER_CONFIGURATION / EVAL_CHUNK_SECONDS)
    for config in ONLINE_CONFIGS:
        trainer = make_trainer(config, seed=7)
        online_trainers[config['label']] = trainer
        measured_training_s = 0.0
        for _ in range(chunks):
            _, chunk_logs, trainer = deep_cfr_timed_loop(
                spec,
                training_seconds=EVAL_CHUNK_SECONDS,
                trainer=trainer,
                traversals_per_player=512,
                eval_every=0,
                exact_averager=None,
                final_eval=False,
                debug=False,
            )
            chunk_df = pd.json_normalize(chunk_logs['training_series'])
            chunk_df['configuration'] = config['label']
            chunk_df['measured training s'] = measured_training_s + chunk_df['elapsed_s']
            measured_training_s += float(chunk_df['elapsed_s'].iloc[-1])
            online_training_rows.extend(chunk_df.to_dict('records'))

            metrics = exact_metrics(trainer)
            online_eval_rows.append({
                'configuration': config['label'],
                'iteration': trainer.iteration,
                'measured training s': measured_training_s,
                **metrics,
            })

online_train_df = pd.DataFrame(online_training_rows)
online_eval_df = pd.DataFrame(online_eval_rows)
display(online_eval_df.tail() if not online_eval_df.empty else online_eval_df)

KeyError: 'compile_models'

In [9]:
if not online_eval_df.empty:
    summary_rows = []
    for label, eval_group in online_eval_df.groupby('configuration'):
        train_group = online_train_df[online_train_df['configuration'] == label]
        summary_rows.append({
            'configuration': label,
            'iterations completed': int(eval_group['iteration'].iloc[-1]),
            'final learned-average exploitability': eval_group['learned average exploitability'].iloc[-1],
            'best learned-average exploitability': eval_group['learned average exploitability'].min(),
            'final current exploitability': eval_group['current exploitability'].iloc[-1],
            'mean traversal s': train_group['timing.traversal_s'].mean(),
            'mean advantage fit s': train_group['timing.advantage_training_s'].mean(),
            'mean strategy fit s': train_group['timing.strategy_training_s'].mean(),
            'advantage records seen': sum(online_trainers[label].advantage_buffers[p].seen for p in (0, 1)),
            'strategy records seen': sum(online_trainers[label].strategy_buffers[p].seen for p in (0, 1)),
        })
    online_summary_df = pd.DataFrame(summary_rows).set_index('configuration')
    display(online_summary_df)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for label, group in online_eval_df.groupby('configuration'):
        axes[0].plot(
            group['measured training s'] / 60,
            group['learned average exploitability'].clip(lower=1e-8),
            marker='o',
            label=label,
        )
        axes[1].plot(
            group['measured training s'] / 60,
            group['current exploitability'].clip(lower=1e-8),
            marker='o',
            label=label,
        )
    for label, group in online_train_df.groupby('configuration'):
        axes[2].plot(
            group['measured training s'] / 60,
            group['timing.traversal_s'] + group['timing.advantage_training_s'] + group['timing.strategy_training_s'],
            label=label,
        )
    axes[0].set_title('Learned-average exact exploitability')
    axes[1].set_title('Current exact exploitability')
    axes[2].set_title('Measured iteration time')
    axes[0].set_yscale('log')
    axes[1].set_yscale('log')
    for ax in axes:
        ax.set_xlabel('measured training minutes')
        ax.legend()
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## Optional Asynchronous Exact Evaluation Pattern

For long runs, use `AsyncDeepCFREvaluator` to submit saved learned-average policies to one CPU worker. The GPU loop can continue immediately; call `collect_ready()` periodically and `wait()` before shutting down. Snapshots are retained under the run directory.

In [ ]:
from liars_poker.eval.deep_cfr_async import AsyncDeepCFREvaluator

# Example for a long run:
# evaluator = AsyncDeepCFREvaluator(REPO_ROOT / 'artifacts' / 'deep_cfr_gpu_run')
# evaluator.submit(trainer.iteration, trainer.average_policy())
# ready_results = evaluator.collect_ready()
# final_results = ready_results + evaluator.wait()
# evaluator.close()
# pd.DataFrame(final_results).sort_values('iter')